# Live folder watching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/watch_folder.ipynb)

During a microscope session, results land in a folder one file at a time. The
viewers can **watch that folder** and grow in place - keep one widget open at
the scope and let new acquisitions appear, instead of re-running a notebook
after every file.

Where this earns its keep:

- **Survey acquisition** - `Show2D.from_folder` adds a panel per new HAADF as
  it lands, so you pick the next field of view from what you already have.
- **In-situ / time series** - `Show3D.from_folder` appends frames to a
  scrubbable stack while the experiment runs.
- **4D-STEM sessions** - `Show4DSTEM.watch_folder` appends each completed
  `*_master.h5` to the dataset slider, skipping files still being written.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/bobleesj/quantem.widget/blob/main/docs/tutorials/watch_folder.ipynb). For finished results, use [HTML and file export](widget_export) to export interactive HTML or share a trusted notebook with widget state.
```

In [ ]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except Exception:
    pass
else:
    from google.colab import output

    output.enable_custom_widget_manager()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/bobleesj/quantem.widget.git"],
        check=True,
    )


In [ ]:
from quantem.widget.datasets import showfolder_gold

folder = showfolder_gold(verbose=False)  # real gold HAADF session

## Watch survey images with Show2D

`Show2D.from_folder` reads every readable image at full resolution and starts
background polling (`watch=True` is the default, `watch_interval=1.0`
seconds). Files added to the folder later become new panels in this same
widget; partially-written files are retried until stable. Here it opens the
real session's overview images:

In [ ]:
import pathlib
import shutil
import tempfile

from quantem.widget import Show2D

session = pathlib.Path(tempfile.mkdtemp(prefix="live_session_"))
for p in sorted(folder.glob("*overview 0[12]*")):
    shutil.copy(p, session / p.name)

viewer = Show2D.from_folder(session)
viewer

With the kernel running, drop another file into the folder and the widget adds
the panel by itself within a poll interval:

```python
shutil.copy(next(folder.glob("*overview 03*")), session)  # a new panel appears
```

On a live session you would point it at the acquisition folder and leave it
open:

```python
viewer = Show2D.from_folder("/data/session", pattern="*.emd")   # keeps growing
viewer = Show2D.from_folder("/data/session", watch=False)       # one-shot read
```

## Watch a growing stack with Show3D

For same-size frames (in-situ series, tilt series, a denoiser writing
results), `Show3D.from_folder` plays the folder as one stack and appends new
frames in place:

```python
from quantem.widget import Show3D

stack = Show3D.from_folder("/data/growth_run", pattern="frame_*.tif")
```

## Watch a 4D-STEM session with Show4DSTEM

For live scope folders, the Show4DSTEM viewer appends each completed
`*_master.h5` into the same dataset slider - detector and scan interactions
stay real-time after every append, and partial files are skipped until ready:

```python
from quantem.widget import load, Show4DSTEM

viewer = Show4DSTEM(load("/data/session/first_master.h5"))
viewer.watch_folder(interval=2.0)
```

See [Show4DSTEM live scope folders](../api/show4dstem.md#live-scope-folders)
for the full workflow, including Apple Silicon notes.

## From the command line

The same watching works without a notebook:

```bash
quantem show2d ./frames/ --watch     # live folder -> appending Show2D
quantem show3d ./frames/ --watch     # live folder -> appending Show3D stack
quantem show4dstem ./masters/        # live master folder -> Show4DSTEM
```

## Related pages

- [ShowFolder session browser](showfolder) - browse and star a finished session
- [IO/GPU](io_gpu) - the loaders these watchers are built on
- [Show4DSTEM tutorial](show4dstem) - virtual detectors on the appended data